In [1]:
import numpy as np
import pandas as pd
import torch

df = pd.DataFrame(
    [
        {"participant": "p1", "task": "A", "trial": 0, "choice": 1, "x1": 0.1},
        {"participant": "p1", "task": "A", "trial": 1, "choice": 0, "x1": 0.2},
        {"participant": "p1", "task": "A", "trial": 2, "choice": 1, "x1": 0.3},

        {"participant": "p2", "task": "A", "trial": 0, "choice": 0, "x1": 1.1},
        {"participant": "p2", "task": "A", "trial": 1, "choice": 1, "x1": 1.2},
        {"participant": "p2", "task": "A", "trial": 2, "choice": 0, "x1": 1.3},

        {"participant": "p1", "task": "B", "trial": 0, "choice": 1, "x1": 2.1},
        {"participant": "p1", "task": "B", "trial": 1, "choice": 1, "x1": 2.2},
        {"participant": "p1", "task": "B", "trial": 2, "choice": 0, "x1": 2.3},

        {"participant": "p2", "task": "B", "trial": 0, "choice": 0, "x1": 3.1},
        {"participant": "p2", "task": "B", "trial": 1, "choice": 0, "x1": 3.2},
        {"participant": "p2", "task": "B", "trial": 2, "choice": 1, "x1": 3.3},
    ]
)

df

,participant,task,trial,choice,x1
0,p1,A,0,1,0.1
1,p1,A,1,0,0.2
2,p1,A,2,1,0.3
3,p2,A,0,0,1.1
4,p2,A,1,1,1.2
5,p2,A,2,0,1.3
6,p1,B,0,1,2.1
7,p1,B,1,1,2.2
8,p1,B,2,0,2.3
9,p2,B,0,0,3.1


In [2]:
def df_to_tensors(df, values_cols, index_cols=None):
    if index_cols is None:
        index_cols = ["participant", "task", "trial"]

    tensors = {}
    for value in values_cols:
        selected_cols = [*index_cols, value]
        tmp_values = df[selected_cols].values
        index_encoding = tuple(np.unique(tmp_values[:, i], return_inverse=True)
                      for i in range(len(index_cols)))
        #print("index_encoding shape:", index_encoding.shape)
        wide_shape = [len(unique_values) for unique_values, _ in index_encoding]
        wide_values = np.full(wide_shape, np.nan)
        wide_idx = tuple(inverse_indices for _, inverse_indices in index_encoding)
        wide_values[wide_idx] = tmp_values[:, -1]
        tensors[value] = torch.from_numpy(wide_values).reshape(-1, wide_values.shape[-1])
         
    return tensors

res = df_to_tensors(df, ["choice", "x1"])

In [4]:
print(res["choice"].shape)
print(res)

torch.Size([4, 3])
{'choice': tensor([[1., 0., 1.],
        [1., 1., 0.],
        [0., 1., 0.],
        [0., 0., 1.]], dtype=torch.float64), 'x1': tensor([[0.1000, 0.2000, 0.3000],
        [2.1000, 2.2000, 2.3000],
        [1.1000, 1.2000, 1.3000],
        [3.1000, 3.2000, 3.3000]], dtype=torch.float64)}


In [6]:
index_cols = ["participant", "task", "trial"]
value_col = "choice"

selected_cols = [*index_cols, value_col]
long_values = df[selected_cols].values

print("selected_cols:")
print(selected_cols)

print("\nlong_values shape:")
print(long_values.shape)

print("\nlong_values:")
print(long_values)

print(type(long_values))

selected_cols:
['participant', 'task', 'trial', 'choice']

long_values shape:
(12, 4)

long_values:
[['p1' 'A' 0 1]
 ['p1' 'A' 1 0]
 ['p1' 'A' 2 1]
 ['p2' 'A' 0 0]
 ['p2' 'A' 1 1]
 ['p2' 'A' 2 0]
 ['p1' 'B' 0 1]
 ['p1' 'B' 1 1]
 ['p1' 'B' 2 0]
 ['p2' 'B' 0 0]
 ['p2' 'B' 1 0]
 ['p2' 'B' 2 1]]
<class 'numpy.ndarray'>


In [ ]:
index_encoding = tuple(np.unique(long_values[:, i], return_inverse=True)
                      for i in range(len(index_cols)))

print(index_encoding)

((array(['p1', 'p2'], dtype=object), array([0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1])), (array(['A', 'B'], dtype=object), array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])), (array([0, 1, 2], dtype=object), array([0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2])))


In [8]:
for col_name, (unique_values, inverse_indices) in zip(index_cols, index_encoding):
    print(f"\nColumn: {col_name}")
    print("unique_values:", unique_values)
    print("inverse_indices:", inverse_indices)


Column: participant
unique_values: ['p1' 'p2']
inverse_indices: [0 0 0 1 1 1 0 0 0 1 1 1]

Column: task
unique_values: ['A' 'B']
inverse_indices: [0 0 0 0 0 0 1 1 1 1 1 1]

Column: trial
unique_values: [0 1 2]
inverse_indices: [0 1 2 0 1 2 0 1 2 0 1 2]


In [9]:
wide_shape = [len(unique_values) for unique_values, _ in index_encoding]

print("wide_shape:")
print(wide_shape)

wide_shape:
[2, 2, 3]


In [10]:
wide_values = np.full(wide_shape, np.nan)

wide_idx = tuple(inverse_indices for _, inverse_indices in index_encoding)

print("wide_idx:")
for name, idx in zip(index_cols, wide_idx):
    print(name, idx)

wide_values[wide_idx] = long_values[:, -1]

print("\nwide_values shape:")
print(wide_values.shape)

print("\nwide_values:")
print(wide_values)

wide_idx:
participant [0 0 0 1 1 1 0 0 0 1 1 1]
task [0 0 0 0 0 0 1 1 1 1 1 1]
trial [0 1 2 0 1 2 0 1 2 0 1 2]

wide_values shape:
(2, 2, 3)

wide_values:
[[[1. 0. 1.]
  [1. 1. 0.]]

 [[0. 1. 0.]
  [0. 0. 1.]]]
